# Fine-tuning com LoRA — Projeto Fichário (RAG vs. Fine-tuning)

Este notebook faz o fine-tuning de um modelo pequeno e open-source usando a técnica LoRA, a partir do dataset de Q&A gerado anteriormente.

**Passos:**
1. Instalar dependências
2. Carregar o modelo base
3. Carregar o dataset de Q&A
4. Configurar o LoRA
5. Treinar
6. Testar o modelo ajustado
7. Salvar / publicar

> Antes de rodar: em **Ambiente de execução > Alterar tipo de ambiente**, selecione GPU (T4).

## 1. Instalar dependências

In [ ]:
!pip install -q transformers peft datasets accelerate bitsandbytes

## 2. Carregar o modelo base

Usamos um modelo pequeno (Phi-3-mini) e carregamos em 4-bit (quantizado) para caber na GPU gratuita do Colab.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

NOME_MODELO = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(NOME_MODELO)
tokenizer.pad_token = tokenizer.eos_token

modelo = AutoModelForCausalLM.from_pretrained(
    NOME_MODELO,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Modelo carregado:", NOME_MODELO)

## 3. Carregar o dataset de Q&A

Faça upload do arquivo `dataset_finetuning.jsonl` (gerado no script anterior) para o Colab antes de rodar esta célula — use o ícone de pasta na barra lateral esquerda.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="dataset_finetuning.jsonl", split="train")

def formatar_exemplo(exemplo):
    mensagens = exemplo["messages"]
    pergunta = mensagens[0]["content"]
    resposta = mensagens[1]["content"]
    texto = f"<|user|>\n{pergunta}<|end|>\n<|assistant|>\n{resposta}<|end|>"
    return {"text": texto}

dataset = dataset.map(formatar_exemplo)
print(dataset[0]["text"])

## 4. Configurar o LoRA

Em vez de ajustar todos os parâmetros do modelo (caro e pesado), o LoRA adiciona pequenas camadas treináveis, mantendo o modelo original congelado.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

modelo = prepare_model_for_kbit_training(modelo)

config_lora = LoraConfig(
    r=8,                      # tamanho das camadas extras (menor = mais leve)
    lora_alpha=16,
    target_modules=["qkv_proj", "o_proj"],  # camadas de atenção do Phi-3
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

modelo = get_peft_model(modelo, config_lora)
modelo.print_trainable_parameters()

## 5. Treinar

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

argumentos_treino = TrainingArguments(
    output_dir="./resultado_finetuning",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=argumentos_treino,
)

trainer.train()

## 6. Testar o modelo ajustado

Compare a resposta do modelo fine-tuned com uma pergunta baseada no seu domínio.

In [ ]:
pergunta_teste = "Qual é o prazo de garantia do produto XPTO-200?"
prompt = f"<|user|>\n{pergunta_teste}<|end|>\n<|assistant|>\n"

entrada = tokenizer(prompt, return_tensors="pt").to(modelo.device)
saida = modelo.generate(**entrada, max_new_tokens=100)

print(tokenizer.decode(saida[0], skip_special_tokens=True))

## 7. Salvar / publicar o modelo

Salva os adaptadores LoRA localmente e, opcionalmente, publica no seu Hugging Face Hub pessoal.

In [ ]:
modelo.save_pretrained("./modelo_finetuned_fichario")
tokenizer.save_pretrained("./modelo_finetuned_fichario")

# Para publicar no Hub (requer login com huggingface-cli login):
# modelo.push_to_hub("seu-usuario/fichario-finetuned")
# tokenizer.push_to_hub("seu-usuario/fichario-finetuned")

print("Modelo salvo em ./modelo_finetuned_fichario")